In [39]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)

In [ ]:
from consensus_aligner import ChromatographicAligner
from ipyfilechooser import FileChooser
from interface_jupyter import Interface
import ipywidgets as widgets


In [ ]:
# selection de files
#c hoix output
# choix seed

class ChromatographicAlignerUI(Interface):
    def __init__(self):
        super().__init__(supported_extensions=('.txt',))
        self._setup_default_parameters()
       
        self._create_parameter_widgets()
        self._create_base_widgets()
        self._create_action_widgets()
        self._create_alignment_widget()
        self._setup_callbacks()
        self._setup_environment()

    def _setup_default_parameters(self):
        """Set up default parameters for the chromatographic alignment."""
        # private , fixed for the moment #TODO
        self._rt1_penalty = 1
        self._rt2_penalty = 5
        self._similarity_cutoff = 90
        self._disimilarity_cutoff = 90
        self._num_cores = 1
        self._missing_value_limit = 0
        self._quant_method= "T"
        self._auto_tune_match_stringency = False
        self._missing_peak_finder_similarity_lax=0.85


    
    def _create_parameter_widgets(self):
        self.w_seedFile = widgets.Text(value='1')
        self._seedFile = self._bold_widget("Seed file", self.w_seedFile)
        self.seed_def = self.create_help_text(
             "File number in inputFileList to initialize alignment."
        )

    def _create_alignment_widget(self):
        self.txt_title = widgets.HTML(value="<H1>Chromatographic Alignment</H1>")
         # NIST matching
        self.nist = widgets.Checkbox(
            value=True,
            description='Enable NIST Database Matching',
            style=self.style,
            disabled=False
        )
              # Action widgets
        self.run_button, self.clear_button, self.output = self._create_action_widgets()

    def _validate_parameters(self):
        errors = []
        if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
            errors.append("Output directory cannot be empty")

        return errors

    def _on_button_click(self, b):
        """Handle button click event."""
        with self.output:
            self.output.clear_output()
            print("Running alignment... ")
            # validate parameter #TODO
            # errors = self._validate_parameters()
            if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
                print("Output directory cannot be empty")
                return
            
            print("\n Collecting files from selections...   ")
            selected_files = self.get_all_files_from_selections()
            if not selected_files:
                print("Please select files or folders containing .txt files.")
                return
            print(f"\n✅ {len(selected_files)} compatible files found")

            output_path = self.get_output_path()    

            try:
                self.aligner = ChromatographicAligner(
                    rt1_penalty=self._rt1_penalty,
                    rt2_penalty=self._rt2_penalty,
                    similarity_cutoff=self._similarity_cutoff,
                    disimilarity_cutoff=self._disimilarity_cutoff,
                    num_cores=self._num_cores,
                    missing_value_limit=self._missing_value_limit,
                    quant_method=self._quant_method,
                    auto_tune_match_stringency=self._auto_tune_match_stringency,
                    missing_peak_finder_similarity_lax=self._missing_peak_finder_similarity_lax
                ) 

                result = self.aligner.consensus_align_bis(selected_files, 
                                                          self.w_seedFile.value - 1, 
                                                        #   self.nist.value,
                                                          common_ions=None,
                                                          standard_library=None,
                                                        )
                self.aligner.save_results(output_path)
            except Exception as e:
                print(f"Error during alignment: {e}")

    def display(self):
        """Display the interface."""
        display(self.txt_title,
                widgets.VBox([self._vbox, self._vbox2]),
                self._seedFile,
                self.seed_def,
                self.nist,
                widgets.HBox([self.run_button, self.clear_button]),
                self.output)


In [44]:
t = ChromatographicAlignerUI()
t.display()

HTML(value='<H1>Chromatographic Alignment</H1>')

HTML(value='\n            <div style="margin-left: 20px; font-style: italic; color: #666; font-size: 0.9em;">\…

Checkbox(value=True, description='Enable NIST Database Matching', style=DescriptionStyle(description_width='in…

Output()

In [ ]:
#choix dun filtre
#choix output

In [ ]:
# # run avec le filtre
# filtered_results = aligner.filter_alignment_matrix(missing_value_threshold=0.5)
# aligner.save_results(output_dir, filtered_results)